#  <center> Problem Set 6 <center>
<center> 3.C01/3.C51, 7.C01/7.C51, 10.C01/10.C51, 20.C01/20.C51<center>



<b>Name:</b>

<b>Kerberos id:</b>

# Install required packages & Mount Google Drive (0 points)

In [27]:
!pip install fair-esm scikit-learn biopython matplotlib torch pandas tqdm transformers

In [28]:
!pip install --upgrade gdown

In [29]:
pip install torch transformers scikit-learn numpy

In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np
from Bio import SeqIO
import seaborn as sns
import csv
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import umap
import matplotlib.pyplot as plt
import pickle
import os
from google.colab import drive
from collections import Counter
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_recall_fscore_support, precision_recall_curve
from tqdm import tqdm
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

In [31]:
# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Get Training & Validation Data (0 points)

Run the cells below to extract the training and validation data.

In [32]:
#get validation data
file_id="1s0gr_uK7gs_jkc1dw-fZGc4XNQQ_51bT"
destination="valid_data.pkl"
!wget --no-check-certificate "https://drive.google.com/uc?export=download&id={file_id}" -O {destination}

--2025-05-05 23:41:39--  https://drive.google.com/uc?export=download&id=1s0gr_uK7gs_jkc1dw-fZGc4XNQQ_51bT
Resolving drive.google.com (drive.google.com)... 142.251.2.139, 142.251.2.113, 142.251.2.101, ...
Connecting to drive.google.com (drive.google.com)|142.251.2.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1s0gr_uK7gs_jkc1dw-fZGc4XNQQ_51bT&export=download [following]
--2025-05-05 23:41:39--  https://drive.usercontent.google.com/download?id=1s0gr_uK7gs_jkc1dw-fZGc4XNQQ_51bT&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.101.132, 2607:f8b0:4023:c06::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.101.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31750082 (30M) [application/octet-stream]
Saving to: ‘valid_data.pkl’

valid_data.pkl      100%[===================>]  30.28M  72.7MB/s  

In [33]:
#get training data
file_id = "1XFObkJHwns-BDmS7O4xmKc35iSwhinNk"
destination = "train_data.pkl"
gdown.download(id=file_id, output=destination, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1XFObkJHwns-BDmS7O4xmKc35iSwhinNk
From (redirected): https://drive.google.com/uc?id=1XFObkJHwns-BDmS7O4xmKc35iSwhinNk&confirm=t&uuid=f6613e03-980f-4277-88f2-519396b88e3a
To: /content/train_data.pkl
100%|██████████| 653M/653M [00:10<00:00, 62.5MB/s]


'train_data.pkl'

In [34]:
sampled_train_data = pd.read_pickle('train_data.pkl')
sampled_valid_data = pd.read_pickle('valid_data.pkl')

# 1.1 Generate and Visualize ESM2 Embeddings (15 points)



Project the ESM2 embeddings to 2 dimensions and visualize them using UMAP. (2.5 points)


* What do you observe in terms of the information captured by the ESM2 embeddings for the training vs. validation sets? (2.5 points)

* How might the observed patterns in the distribution of training vs. validation data affect out-of-distribution testing and generalization performance? (5 points)

* What information about proteins can be useful when doing a train-test-validation split on protein sequences? (5 points)

In [ ]:
#TODO#

# 1.2 Extract and binarize GO annotations for training and validation data (5 points)

Run the cells below to extract the GO terms (labels) for the training data, binarize the labels using `MultiLabelBinarizer`, and print the total number of unique GO terms in the dataset. Then, complete the preprocessing of the GO terms for the validation data using the `MultiLabelBinarizer.transform` function.

In [ ]:
if isinstance(sampled_train_data['annotations'].iloc[0], str):
    sampled_train_data['annotations'] = sampled_train_data['annotations'].apply(ast.literal_eval)

all_go_terms = sorted(set(term for go_list in sampled_train_data['annotations'] for term in go_list))
print(f"Number of unique GO terms is {len(all_go_terms)}")

mlb = MultiLabelBinarizer(classes=all_go_terms)
train_go_annotations = mlb.fit_transform(sampled_train_data['annotations'])
sampled_train_data['go_annotations'] = list(train_go_annotations)



# Filter out GO terms that are not in mlb class defined above for training set so train and validation set share labels
if isinstance(sampled_valid_data['annotations'].iloc[0], str):
    sampled_valid_data['annotations'] = sampled_valid_data['annotations'].apply(ast.literal_eval)

known_go_terms = set(mlb.classes_)

sampled_valid_data['filtered_annotations'] = sampled_valid_data['annotations'].apply(
    lambda go_list: [term for term in go_list if term in known_go_terms])


#TODO#

Number of unique GO terms is 81806


# 1.3 Creating datasets for model training and validation (5 points)

Complete the definition of the `ProteinDataset` class shown below. Both the ESM2 embeddings and labels should be converted to torch.float32 (2.5 points). Then create datasets and dataloaders for the training and validation data using the `ProteinDataset` class you defined. Don't forget to shuffle the training dataloader but not the validation dataloader (2.5 points).

In [ ]:
batch_size=32

class ProteinDataset(Dataset):
    def __init__(self, dataframe, go_annotations):

        ## TODO ##

    def __len__(self):
      return len(self.embeddings)

    def __getitem__(self, idx):
      self.embeddings[idx], self.labels[idx]

IndentationError: expected an indented block after function definition on line 4 (<ipython-input-81-3d95d98372c4>, line 8)

In [ ]:
## TODO ##
train_dataset =
valid_dataset =

train_loader =
valid_loader =

# 1.4 Define MLP (5 points)

Complete the definition of the ESM2MLP class with the right dimensions. (5 points)

In [ ]:
class MLPBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.dense = nn.Linear(input_dim, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        # First block
        if not hasattr(self, 'is_residual'):
            return F.relu(self.layer_norm(self.dropout(self.dense(x))))

        # Second block with residual connection
        identity = x
        x = self.dense(x)
        x = self.layer_norm(x)
        x = F.relu(x)
        return identity + x \


class ESM2MLP(nn.Module):
    def __init__(self, num_classes=?):  #TODO
        super().__init__()
        self.blocks = nn.Sequential(
            MLPBlock(?, 1024),  #TODO
            MLPBlock(1024, 1024),
            nn.Linear(1024, num_classes))
        self.blocks[1].is_residual = True

    def forward(self, x):
        return self.blocks(x)


#Instantiate model
model = ESM2MLP()

SyntaxError: invalid syntax (<ipython-input-47-a2218b3a7068>, line 22)

# 1.5.1 Train Model (20 points)

To train the model, start by setting up the Adam optimizer, with a learning rate `lr` of `1e-4`. (5 points)

Next, write your training loop using the provided `train_epoch()` function, and train the model for 10 epochs. This is a classification problem, so you will want to compute the binary cross-entropy loss using `nn.BCEWithLogitsLoss()`, which requires you to input classification probabilities and the ground truth labels from your data. (10 points)


Record the loss across epochs and plot the training loss across epochs. Comment briefly on the trend you observe. (5 points)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in tqdm(loader, desc="Training"):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ESM2MLP(
  (blocks): Sequential(
    (0): MLPBlock(
      (dense): Linear(in_features=2560, out_features=1024, bias=True)
      (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): MLPBlock(
      (dense): Linear(in_features=1024, out_features=1024, bias=True)
      (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (2): Linear(in_features=1024, out_features=81806, bias=True)
  )
)

In [ ]:
##TODO##

# 1.5.2 Evaluate Model (10 points)

To evaluate the model, run the code block below.

Why do we need the sigmoid activation when looking at the predictions and converting them to probabilities? (5 points)

Compute Precision, Recall, and F1-Score using the relevant scikit-learn function and comment on the values. Note the 'average' argument. (5 points)

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(valid_dataset.embeddings.to(device))
    probabilities = torch.sigmoid(logits).cpu().numpy()

# Binary predictions with 0.5 threshold
binary_predictions = (probabilities >= 0.5).astype(np.float32)

# Get top 5 predicted indices per sample
top_hits = np.argsort(-probabilities, axis=1)[:, :5]

# Ensure top-5 predictions are always marked as 1
for i, indices in enumerate(top_hits):
    binary_predictions[i, indices] = 1

# Convert indices to GO term strings
top_go_terms = [[mlb.classes_[idx] for idx in row] for row in top_hits]
sampled_valid_data['top_5_predicted_go_terms'] = top_go_terms

In [ ]:
##TODO##

# 1.6 Compare with MLP trained on one-hot encoded sequences (30 points)




We will compare the performance of an **MLP trained on ESM2 embeddings** with a simpler **MLP trained on one-hot encoded sequence data** using the validation data.

Start by one-hot encoding the sequence data using the `one_hot_encode_sequence` function that you will have to complete (5 points). Then, complete the `ProteinOneHotDataset` class and create the training and validation dataloaders (5 points). Make sure to use the `collate_fn` to handle variable-length sequences when defining the DataLoader.

Implement the small MLP with the following architecture (5 points):

To do so, define the `__init__` function of the `OneHotMLP` class with `hidden_dim = 256`. You will also need to define `input_dim` and `num_classes`.

The class should inherit from `nn.Module`, and use these layers to build the network:
- `nn.Linear`
- `nn.ReLU`
- `nn.Dropout`
- `nn.Sequential`

The feedforward neural network should:
1. Take an input of shape `(batch_size, input_dim)`.
2. Have a hidden layer with **256** nodes and ReLU activation.
3. Map the output to **num_classes** labels.
4. Apply a **0.1** dropout rate after ReLU.

The `forward` method is already implemented for you.

In [ ]:
#One-hot encode sequences
amino_acids = "ACDEFGHIKLMNPQRSTVWY"
aa_to_int = {aa: i for i, aa in enumerate(amino_acids)}

def one_hot_encode_sequence(seq, aa_to_int):

      ##TODO##

    return one_hot

##TODO##

In [ ]:
#Define Dataset Class and Dataloders

batch_size=8

class ProteinOneHotDataset(Dataset):
    def __init__(self, dataframe, go_annotations):
        ##TODO##

    def __len__(self):
        ##TODO##

    def __getitem__(self, idx):
        ##TODO##
        return sequence, label


def collate_fn(batch):
    sequences, labels = zip(*batch)
    sequences = pad_sequence(sequences, batch_first=True)  # Pad sequences to same length
    labels = torch.stack(labels)
    return sequences, labels


##TODO##

train_dataset =
valid_dataset =
train_loader =
valid_loader =

In [ ]:
#Define and call MLP


class OneHotMLP(nn.Module):

    ##TODO##

    def forward(self, x):
        x = x.mean(dim=1)  #mean-pooling across the sequence length dimension
        return self.network(x) #pass the averaged sequence through the model

model = OneHotMLP()

Train the model using the same hyperparameters as defined above (5 points).


In [ ]:
#Train model

##TODO##

Evaluate the model on the validation data. Use a threshold of 0.5 to compute the binary predictions. (5 points)

In [ ]:
##TODO##

Compute Precision, Recall, and F1-Score as you did above. Comment on the values and compare performances of the MLP trained on one-hot encodings vs. MLP trained on ESM2 embeddings, and explain potential reasons for differences in performance (2.5 points). In your answer, make sure to comment on potential issues with one-hot encodings as well as on the information content of ESM2 embeddings. (2.5 points)

In [ ]:
##TODO##

# 2. Investigating a protein of interest: Mouse hypothalamic galanin-like neuropeptide (GALP_MOUSE) (5 points)

Compare the GO terms predicted by the MLP and naive approach for `GALP_MOUSE` in the validation set and its experimental annotations.

Note: you can ignore '|IEA, '|IDA' suffixes in the GO terms. These relate to how the annotations were made - more [here](http://geneontology.org/GO_REF/0000028.html).

Use this [link](https://www.ebi.ac.uk/QuickGO/) to look up the GO terms.

In [ ]:
#TODO#

# 3. Predicting Protein Structure using ESMFold (5 points)



Use the relevant sequence from the validation dataframe to predict the 3D structure of `GALP_MOUSE` using [ESMFold](https://esmatlas.com/resources?action=fold).  

Paste the relevant protein sequence and hit **"Fold Sequence"**.  

Please upload below a screenshot of the top predicted structure using the **Files** tab to the left.  

Include the sequence you pasted into ESMFold in your answer below and comment on the confidence of the top prediction.  

In [ ]:
#TODO#